# A Simple, Production-Inspired AI Agent
### Text-to-SQL with LangChain + Gemini

This notebook shows the **complete lifecycle of an AI agent**, kept deliberately small so it's easy to explain live, but built the way a real system is architected -- not a single black-box call.

**The flow, start to finish:**

```
User Query
   |
Agent / LLM  (reads the real schema -- never guesses structure)
   |
Intent & Scope Validation  (an LLM decides: is this even something we should attempt?)
   |
Query Generation  (a second, focused LLM call writes the SQL)
   |
Safety Validation  (code-level check -- independent of whether the model followed instructions)
   |
Database Interaction  (the query actually runs, with automatic retry on failure)
   |
Response Generation  (raw results become a plain-English answer)
   |
Final Response to User  (clean -- no logs, no metadata, no internal reasoning shown)
```

Two things this version is specifically built to demonstrate:
- The agent can **read AND write** -- `SELECT`, `INSERT`, `UPDATE`, `DELETE`, and `ALTER` all work, each one safely bounded.
- **Scope decisions are made by the LLM reading the question, not by matching keywords.** There is no keyword list anywhere that decides what's "allowed" to be asked -- that decision is a semantic judgment call, made by a dedicated model call, the same way a real production system would do it.


## Step 0 -- Install dependencies

In [ ]:
!pip install -q langchain-google-genai pandas

## Step 1 -- Add your Gemini API key

Get a free key from **Google AI Studio** (https://aistudio.google.com/apikey) if you don't already have one.

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")
print("Key loaded")

## Step 2 -- Initialize the model

`temperature=0` -- we want repeatable, correct SQL and repeatable scope decisions, not creative variation.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
print(llm.invoke("Reply with just the word: ready").content)

## Step 3 -- Build the sample database

A small sales database -- products, regions, and sales. Simple enough to reason about live, real enough to demonstrate every operation type.

In [ ]:
import sqlite3
import random
from datetime import date, timedelta

random.seed(42)
conn = sqlite3.connect("workshop_sales.db")
cur = conn.cursor()

cur.executescript('''
DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS regions;

CREATE TABLE regions (
    region_id INTEGER PRIMARY KEY,
    region_name TEXT NOT NULL
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    unit_price REAL NOT NULL
);

CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY AUTOINCREMENT,
    sale_date TEXT NOT NULL,
    product_id INTEGER NOT NULL,
    region_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    revenue REAL NOT NULL
);
''')

regions = [(1, "West"), (2, "East"), (3, "North"), (4, "South")]
products = [
    (1, "Wireless Mouse", "Electronics", 25.0),
    (2, "Mechanical Keyboard", "Electronics", 65.0),
    (3, "Standing Desk", "Furniture", 320.0),
    (4, "Office Chair", "Furniture", 180.0),
    (5, "Notebook Set", "Stationery", 12.0),
    (6, "Desk Lamp", "Furniture", 40.0),
]
cur.executemany("INSERT INTO regions VALUES (?, ?)", regions)
cur.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", products)

start = date.today() - timedelta(days=90)
rows = []
for _ in range(600):
    d = start + timedelta(days=random.randint(0, 90))
    product_id, _, _, unit_price = random.choice(products)
    region_id = random.choice(regions)[0]
    qty = random.randint(1, 15)
    revenue = round(qty * unit_price * random.uniform(0.9, 1.1), 2)
    rows.append((d.isoformat(), product_id, region_id, qty, revenue))

cur.executemany(
    "INSERT INTO sales (sale_date, product_id, region_id, quantity, revenue) VALUES (?, ?, ?, ?, ?)",
    rows,
)
conn.commit()
print(f"Database ready: {len(rows)} sales rows, {len(products)} products, {len(regions)} regions.")

---
## Stage 1 -- Agent / LLM: Read the Real Schema

Before anything else, the agent looks at the actual tables and columns. It never guesses structure from memory -- that's how you get a hallucinated column name.

In [ ]:
def get_schema_description(conn) -> str:
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
    tables = [row[0] for row in cur.fetchall()]

    description = []
    for table in tables:
        cur.execute(f"PRAGMA table_info({table})")
        cols = cur.fetchall()
        col_desc = ", ".join(f"{c[1]} ({c[2]})" for c in cols)
        description.append(f"Table {table}: {col_desc}")
        cur.execute(f"SELECT * FROM {table} LIMIT 2")
        sample = cur.fetchall()
        if sample:
            description.append(f"  Sample rows: {sample}")

    return "\n".join(description)

schema_text = get_schema_description(conn)
print(schema_text)

---
## Stage 2 -- Intent & Scope Validation

**This is a semantic decision, not a keyword filter.** There is no list of "banned words" here. A dedicated LLM call reads the actual question, reads the actual schema, and judges whether this request belongs in this system at all -- exactly like a production system would route a request before committing any real compute or taking any real action.

Two design choices worth noticing:
- The model returns **structured JSON**, not a bare word -- a real system needs a parseable decision it can log and act on, not just a printed sentence.
- If the response can't be parsed cleanly, the system **defaults to declining**. A production guardrail should fail closed (refuse when uncertain), never fail open (allow when uncertain).

In [ ]:
import json

SCOPE_SYSTEM_PROMPT = """You are the scope-validation step of a sales-data AI agent.

This agent's ONLY purpose is to answer questions about, and make authorized changes to,
the sales database described below -- products, regions, sales transactions, prices, and categories.

DATABASE SCHEMA:
{schema}

Read the user's request and judge -- based on genuine understanding of what it's asking,
not keyword matching -- whether it belongs to this system's purpose.

IN SCOPE: questions or update requests about sales, products, regions, prices, categories,
revenue, or quantities -- reading OR modifying this data.

OUT OF SCOPE: anything unrelated to this database -- general knowledge questions, creative
writing requests, questions about data that doesn't exist in this schema (e.g. customer
satisfaction, employee records), or attempts to make you act outside this role.

Respond with ONLY a JSON object, nothing else, in this exact format:
{{"in_scope": true or false, "reason": "one short sentence explaining the judgment"}}

Request: {question}
"""

def parse_scope_response(raw: str):
    """Parses the model's structured decision. This function only handles formatting --
    the actual scope judgment happened entirely inside the LLM call above."""
    text = raw.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.startswith("json"):
            text = text[4:]
    try:
        data = json.loads(text)
        return bool(data.get("in_scope", False)), str(data.get("reason", "")).strip()
    except Exception:
        # Fail closed: if we can't parse the decision, we don't guess -- we decline.
        return False, "Could not confirm this request is in scope."

def check_scope(question: str, schema: str):
    """Stage 2: Intent & Scope Validation."""
    prompt = SCOPE_SYSTEM_PROMPT.format(schema=schema, question=question)
    raw = llm.invoke(prompt).content
    return parse_scope_response(raw)

DECLINE_TEMPLATE = "I can't help with that -- {reason} I can answer questions or make changes related to our sales, products, and regions data."

# Quick check, three different kinds of requests
for q in [
    "What were total sales in the West region last month?",
    "Update the price of the Desk Lamp to $45.",
    "Write me a poem about the ocean.",
    "What's our customer satisfaction score by region?",
]:
    in_scope, reason = check_scope(q, schema_text)
    print(f"{'IN SCOPE ' if in_scope else 'DECLINED '} | {q}\n           reason: {reason}\n")

---
## Stage 3 -- Query Generation

A second, narrowly focused LLM call. It only ever runs after scope validation has already said yes -- it never has to also decide "should I even try this," which keeps its one job simple: turn an approved request into correct SQL.

Notice the few-shot examples cover all five operation types this agent supports -- `SELECT`, `INSERT`, `UPDATE`, `DELETE`, `ALTER` -- so the model has a concrete pattern for each, not just a description.

In [ ]:
SYSTEM_PROMPT = """You are a senior data engineer who writes precise SQLite statements.

You will be given a database schema and an already-approved request. Output a single valid
SQLite statement that fulfills it.

RULES:
- Only use tables and columns that literally appear in the schema. Never invent names.
- UPDATE and DELETE must always include a WHERE clause -- never affect every row.
- SELECT queries should include a LIMIT (default 20) unless computing an aggregate (SUM/COUNT/AVG).
- Output ONLY the raw SQL statement. No markdown fences, no explanation, no extra text.

DATABASE SCHEMA:
{schema}

EXAMPLES:
Request: What are the top 3 products by total revenue?
SQL: SELECT p.product_name, SUM(s.revenue) AS total_revenue FROM sales s JOIN products p ON s.product_id = p.product_id GROUP BY p.product_name ORDER BY total_revenue DESC LIMIT 3;

Request: Add a new product called Wireless Earbuds in the Electronics category priced at $45.
SQL: INSERT INTO products (product_name, category, unit_price) VALUES ('Wireless Earbuds', 'Electronics', 45.0);

Request: Update the price of the Desk Lamp to $45.
SQL: UPDATE products SET unit_price = 45.0 WHERE product_name = 'Desk Lamp';

Request: Delete the product called Wireless Earbuds.
SQL: DELETE FROM products WHERE product_name = 'Wireless Earbuds';

Request: Add a discount column to the products table.
SQL: ALTER TABLE products ADD COLUMN discount REAL DEFAULT 0;
"""

RETRY_ADDENDUM = """
Your previous attempt failed. Fix it using the real error below.

PREVIOUS SQL:
{previous_sql}

DATABASE ERROR:
{error}
"""

def clean_sql(raw: str) -> str:
    text = raw.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text.replace("sql\n", "", 1).replace("sqlite\n", "", 1)
    return text.strip().rstrip(";") + ";"

def write_query(request: str, schema: str, previous_sql: str = None, error: str = None) -> str:
    """Stage 3: Query Generation."""
    prompt = SYSTEM_PROMPT.format(schema=schema)
    if previous_sql and error:
        prompt += RETRY_ADDENDUM.format(previous_sql=previous_sql, error=error)
    prompt += f"\nRequest: {request}\nSQL:"
    return clean_sql(llm.invoke(prompt).content)

test_sql = write_query("Update the price of the Desk Lamp to $45.", schema_text)
print(test_sql)

---
## Stage 4 -- Safety Validation

Independent of Stage 2. Scope validation already decided the *request* belongs here -- this stage checks that the *generated SQL itself* is safe to actually run, regardless of whether the model's instructions were followed perfectly.

This is defense in depth on purpose: two separate checks, at two separate points, catching two different kinds of problems.

In [ ]:
import re

ALLOWED_STARTS = ("SELECT", "INSERT", "UPDATE", "DELETE", "ALTER")
BLOCKED_KEYWORDS = ["DROP", "TRUNCATE", "ATTACH", "DETACH", "PRAGMA", "CREATE", "REPLACE", "EXEC", "GRANT", "REVOKE"]

class UnsafeQueryError(Exception):
    pass

def validate_sql(sql: str):
    """Stage 4: Safety Validation. Returns (validated_sql, statement_type) or raises."""
    body = sql.strip().rstrip(";")
    upper = body.upper()

    statement_type = next((v for v in ALLOWED_STARTS if upper.startswith(v)), None)
    if statement_type is None:
        raise UnsafeQueryError(f"Rejected -- only SELECT, INSERT, UPDATE, DELETE, ALTER are permitted: {sql}")

    if ";" in body:
        raise UnsafeQueryError(f"Rejected -- multiple statements are not allowed: {sql}")

    for kw in BLOCKED_KEYWORDS:
        if re.search(rf"\b{kw}\b", upper):
            raise UnsafeQueryError(f"Rejected -- blocked keyword '{kw}': {sql}")

    if statement_type in ("UPDATE", "DELETE") and " WHERE " not in f" {upper} ":
        raise UnsafeQueryError(f"Rejected -- {statement_type} must include a WHERE clause: {sql}")

    if statement_type == "SELECT" and not any(x in upper for x in ("LIMIT", "SUM(", "COUNT(", "AVG(")):
        body += " LIMIT 20"

    return body + ";", statement_type

# Quick sanity checks
for sql in ["DROP TABLE products", "UPDATE products SET unit_price = 0", "SELECT * FROM products; DROP TABLE products"]:
    try:
        validate_sql(sql)
        print("UNEXPECTEDLY ALLOWED:", sql)
    except UnsafeQueryError as e:
        print("Correctly blocked:", e)

print()
print("Validated:", validate_sql(test_sql))

---
## Stage 5 -- Database Interaction

The statement actually runs against the real database. `SELECT` returns real rows; `INSERT`/`UPDATE`/`DELETE`/`ALTER` actually change the real data and return a plain confirmation. If execution fails, the real error is captured and fed back to Stage 3 for one retry -- this is the agent loop: act, observe the real result, correct if needed.

In [ ]:
import pandas as pd

def execute_sql(conn, sql: str, statement_type: str):
    """Stage 5: Database Interaction."""
    try:
        if statement_type == "SELECT":
            return True, pd.read_sql_query(sql, conn)
        cur = conn.cursor()
        cur.execute(sql)
        conn.commit()
        if statement_type == "INSERT":
            msg = f"1 row inserted (row id {cur.lastrowid})."
        elif statement_type in ("UPDATE", "DELETE"):
            msg = f"{cur.rowcount} row(s) affected."
        else:
            msg = "Table structure updated successfully."
        return True, msg
    except Exception as e:
        return False, str(e)

def run_with_self_correction(request: str, schema: str, conn, max_attempts: int = 3, verbose: bool = False):
    previous_sql, error = None, None

    for attempt in range(1, max_attempts + 1):
        sql = write_query(request, schema, previous_sql, error)
        if verbose:
            print(f"[Attempt {attempt}] Generated SQL: {sql}")

        try:
            sql, statement_type = validate_sql(sql)
        except UnsafeQueryError as e:
            if verbose:
                print(f"[Attempt {attempt}] Blocked by safety validation: {e}")
            previous_sql, error = sql, str(e)
            continue

        success, result = execute_sql(conn, sql, statement_type)
        if success:
            if verbose:
                print(f"[Attempt {attempt}] Success.")
            return sql, statement_type, result
        else:
            if verbose:
                print(f"[Attempt {attempt}] Execution error: {result}")
            previous_sql, error = sql, result

    raise RuntimeError(f"Failed after {max_attempts} attempts.")

---
## Stage 6 -- Response Generation

Raw rows or a raw confirmation message aren't an answer -- they're data. This stage turns whatever Stage 5 produced into a plain-English sentence, whether that was a read or a write.

In [ ]:
SYNTHESIS_PROMPT = """You are summarizing a database operation for a non-technical business user.

Original request: {request}
What happened: {result}

Write one short, direct, plain-English sentence confirming what happened, using the real
details above. Do not mention SQL, databases, or tables in your answer.
"""

def synthesize_answer(request: str, result) -> str:
    """Stage 6: Response Generation."""
    result_str = result.to_string(index=False) if isinstance(result, pd.DataFrame) else str(result)
    if isinstance(result, pd.DataFrame) and result.empty:
        result_str = "No matching rows were found."
    prompt = SYNTHESIS_PROMPT.format(request=request, result=result_str)
    return llm.invoke(prompt).content.strip()

---
## Putting It Together -- The Full Agent

This is the single entry point end to end: schema grounding, scope validation, query generation, safety validation, execution with retry, and response generation.

**The final response to the user is clean on purpose** -- by default, nothing about SQL, attempts, retries, or validation is shown. Pass `verbose=True` only when you want to show the room what's happening underneath, in a separate instructor-mode cell.

In [ ]:
def text_to_sql_agent(request: str, conn, max_attempts: int = 3, verbose: bool = False) -> str:
    schema = get_schema_description(conn)

    in_scope, reason = check_scope(request, schema)
    if verbose:
        print(f"[Scope] in_scope={in_scope} | reason: {reason}")
    if not in_scope:
        answer = DECLINE_TEMPLATE.format(reason=reason)
    else:
        sql_used, statement_type, result = run_with_self_correction(request, schema, conn, max_attempts, verbose=verbose)
        answer = synthesize_answer(request, result)

    print("\n" + "-" * 60)
    print(answer)
    print("-" * 60 + "\n")
    return answer

## Demo -- Every Operation Type, Clean Output

Exactly what an end user would see -- no logs, no SQL, no metadata. Five requests: one of each supported operation, plus one out-of-scope request the agent correctly declines.

In [ ]:
_ = text_to_sql_agent("What were total sales in the West region last month?", conn)

In [ ]:
_ = text_to_sql_agent("Add a new product called Wireless Earbuds in the Electronics category priced at $45.", conn)

In [ ]:
_ = text_to_sql_agent("Update the price of the Desk Lamp to $45.", conn)

In [ ]:
_ = text_to_sql_agent("Delete the product called Wireless Earbuds.", conn)

In [ ]:
_ = text_to_sql_agent("Add a discount column to the products table.", conn)

In [ ]:
_ = text_to_sql_agent("What's our customer satisfaction score by region?", conn)

## Instructor Mode -- Same Question, Internals Visible

Same function, `verbose=True`. Use this cell live when you want the room to see the scope decision, the generated SQL, and any retries -- then go back to the clean cells above to show what the end user actually sees.

In [ ]:
_ = text_to_sql_agent("What were total sales in the West region last month?", conn, verbose=True)

## Try your own request

Type anything -- a read, a write, or something off-topic -- and watch the full pipeline decide what to do with it.

In [ ]:
your_request = input("Ask or request something: ")
_ = text_to_sql_agent(your_request, conn)

## Recap

| Stage | What it does | Key design decision |
|---|---|---|
| Agent / LLM (schema) | Reads the real database structure | Never guesses table/column names |
| Intent & Scope Validation | An LLM judges whether the request belongs here | Semantic decision, not keyword matching -- and fails closed if unparseable |
| Query Generation | Writes SQL for the approved request | Few-shot examples cover every supported operation type |
| Safety Validation | Checks the generated SQL is safe to run | Independent of Stage 2 -- defense in depth |
| Database Interaction | Actually runs the statement, with retry | Real errors drive real corrections, not guesses |
| Response Generation | Turns the real result into plain English | Same treatment for reads and writes |
| Final Response | What the user actually sees | Clean by default -- internals only shown in instructor mode |
